# 消息

在 LangChain 中，消息是模型的基本上下文单元。它们代表模型的输入和输出，携带与 LLM 交互时表示对话状态所需的内容和元数据。

消息是包含以下内容的对象：
- 角色- 标识消息类型（例如system，user）
- 内容——指消息的实际内容（例如文本、图像、音频、文档等）。
- 元数据- 可选字段，例如响应信息、消息 ID 和令牌使用情况

LangChain 提供了一种适用于所有模型提供程序的标准消息类型，确保无论调用哪个模型，行为都保持一致。

## 基本用法
使用消息的最简单方法是创建消息对象，并在调用时将它们传递给模型。


In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

model = init_chat_model("gpt-5-nano")

system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage

### 文本提示
文本提示是字符串——非常适合简单的生成任务，无需保留对话历史记录。

In [ ]:
response = model.invoke("Write a haiku about spring")

在以下情况下使用文本提示：
- 您有一个独立的请求。
- 你不需要对话记录
- 您希望代码复杂度尽可能低。

### 消息提示
或者，您可以通过提供消息对象列表，将消息列表传递给模型。

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)

在以下情况下使用消息提示：
- 管理多轮对话
- 处理多模态内容（图像、音频、文件）
- 包括系统说明
​


### 字典格式
您还可以直接以 OpenAI 聊天自动补全格式指定消息。


In [ ]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]
response = model.invoke(messages)

## 消息类型
- 系统消息——告诉模型如何运行，并为交互提供上下文。
- 人类消息——代表用户输入以及与模型的交互
- AI消息——模型生成的响应，包括文本内容、工具调用和元数据
- 工具消息- 表示工具调用的输出
​


### 系统消息
ASystemMessage代表一组初始指令，用于启动模型的行为。您可以使用系统消息来设定基调、定义模型的角色并建立响应准则。

In [ ]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)

### 人类信息
AHumanMessage代表用户输入和交互。它们可以包含文本、图像、音频、文件以及任何其他数量的多模态内容。

In [ ]:
response = model.invoke([
  HumanMessage("What is machine learning?")
])


human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

### 人工智能消息
表示AIMessage模型调用的输出。它们可以包含多模态数据、工具调用和特定于提供商的元数据，您可以稍后访问这些元数据。

In [ ]:
response = model.invoke("Explain AI")
print(type(response))  # <class 'langchain_core.messages.AIMessage'>

AIMessage调用模型时会返回对象，响应中包含所有相关的元数据。
提供商对消息类型的权重/上下文处理方式不同，这意味着有时手动创建一个新AIMessage对象并将其插入消息历史记录中会很有帮助，就像它来自模型一样。

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)

- text   
​正文内容。


- content   
信息的原始内容。​


- content_blocks    
消息的标准化内容块。
​
- tool_calls  dict[] | None    
模型调用的工具列表。如果没有调用任何工具，则为空。
​
- id  
消息的唯一标识符（由 LangChain 自动生成或在提供程序响应中返回）
​
- usage_metadata   
​消息的使用元数据，其中可能包含令牌计数

- response_metadata    
消息的响应元数据。


### 工具调用
当模型进行工具调用时，它们包含在AIMessage：

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

### Token使用
可以`AIMessage`在其字段中保存令牌计数和其他使用元数据`usage_metadata`：



In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

response = model.invoke("Hello!")
response.usage_metadata

```josn
{'input_tokens': 8,
 'output_tokens': 304,
 'total_tokens': 312,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 256}}

```

UsageMetadata详情请见此处。

### 流和分块
在流式传输过程中，您将收到AIMessageChunk可以组合成完整消息对象的对象：



In [ ]:
chunks = []
full_message = None
for chunk in model.stream("Hi"):
    chunks.append(chunk)
    print(chunk.text)
    full_message = chunk if full_message is None else full_message + chunk

### 工具消息
对于支持工具调用的模型，AI消息可以包含工具调用。工具消息用于将单次工具执行的结果传递回模型。

工具可以直接生成ToolMessage对象。下面我们展示一个简单的示例。更多信息请参阅工具指南。



In [ ]:
# After a model makes a tool call
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

## 消息内容
您可以将消息内容理解为发送到模型的数据有效载荷。消息具有一个content弱类型属性，支持字符串和无类型对象列表（例如字典）。这使得 LangChain 聊天模型能够直接支持提供者原生结构，例如多模态内容和其他数据。
此外，LangChain 还为文本、推理、引用、多模态数据、服务器端工具调用和其他消息内容提供了专用的内容类型。请参见下面的内容块。
LangChain聊天模型接受content属性中的消息内容，并且可以包含：
- 一根弦
- 提供商原生格式的内容块列表
- LangChain标准内容块列表

下面给出一个使用多模态输入的示例：



In [ ]:
from langchain.messages import HumanMessage

# String content
human_message = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

### 标准内容块
LangChain 为消息内容提供了一种跨提供商通用的标准表示形式。

消息对象实现了一个content_blocks属性，该属性会将属性值延迟解析content为标准的、类型安全的表示形式。例如，ChatAnthropic或ChatOpenAI生成的消息将包含相应提供商格式的 `<Message>`thinking或reasoning`<Message>` 块，但可以延迟解析为一致的ReasoningContentBlock表示形式：


In [ ]:
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {"type": "thinking", "thinking": "...", "signature": "WaUjzkyp..."},
        {"type": "text", "text": "..."},
    ],
    response_metadata={"model_provider": "anthropic"}
)
message.content_blocks

In [1]:
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {
            "type": "reasoning",
            "id": "rs_abc123",
            "summary": [
                {"type": "summary_text", "text": "summary 1"},
                {"type": "summary_text", "text": "summary 2"},
            ],
        },
        {"type": "text", "text": "...", "id": "msg_abc123"},
    ],
    response_metadata={"model_provider": "openai"}
)
message.content_blocks

[{'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'summary 1'},
 {'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'summary 2'},
 {'type': 'text', 'text': '...', 'id': 'msg_abc123'}]

请参阅集成指南，开始使用您选择的推理提供程序。

如果 LangChain 之外的应用程序需要访问标准内容块表示，您可以选择将内容块存储在消息内容中。

为此，您可以设置LC_OUTPUT_VERSION环境变量v1。或者，使用以下命令初始化任何聊天模型output_version="v1"：



In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano", output_version="v1")

### 多模态
多模态是指处理不同形式数据的能力，例如文本、音频、图像和视频。LangChain 包含可供不同提供商使用的标准数据类型。

聊天模型可以接受多模态数据作为输入，并生成相应的输出。以下是一些包含多模态数据的输入消息示例。


额外的键可以包含在内容块的顶层，也可以嵌套在内容块中"extras": {"key": value}。

例如，OpenAI和AWS Bedrock Converse 都需要为 PDF 文件指定文件名。 有关详细信息，请参阅您所选模型的提供商页面。


#### 图像输入

In [ ]:
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "image/jpeg",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "file_id": "file-abc123"},
    ]
}

### PDF文档输入



In [ ]:
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "url": "https://example.com/path/to/document.pdf"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {
            "type": "file",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "application/pdf",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "file_id": "file-abc123"},
    ]
}

### 音频输入

In [ ]:
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {
            "type": "audio",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "audio/wav",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {"type": "audio", "file_id": "file-abc123"},
    ]
}

### 视频输入

In [ ]:
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {
            "type": "video",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "video/mp4",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {"type": "video", "file_id": "file-abc123"},
    ]
}